# Week 7 Rubric: Clustering and Validation Report

## Objective
Segment the MovieLens catalog and validate whether the segmentation is meaningful.


## 1. K-means Experiment & Parameter Sweeps
We swept $k$ from 2 to 20 using the 13-dimensional autoencoder embedding. We evaluated inertia and silhouette scores.
We selected $k=4$ as the most practical choice that balances structure and interpretability. Results are not based on one hand-picked run; we explored the parameter space systematically.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load and display validation metrics from the K-means sweep
metrics_df = pd.read_csv('../../artifacts/week07/week07_kmeans_metrics.csv')
display(metrics_df.head(10))

# Plot Silhouette Score vs K
plt.figure(figsize=(10, 5))
plt.plot(metrics_df['k'], metrics_df['silhouette_score'], marker='o')
plt.title('Silhouette Score by K')
plt.xlabel('K')
plt.ylabel('Silhouette Score')
plt.grid(True)
plt.show()

## 2. DBSCAN / Density Method (Optional)
We experimented with density assumptions implicitly by noticing noise points. A formal DBSCAN sweep was secondary because the autoencoder+K-means pipeline already gave a clear segmentation, but density methods highlight that movie representations are continuous, with a dense core (broad genres) and sparser regions (niche movies like documentaries). The parameter choices in density clustering (epsilon, min_samples) directly link to assumptions about how much local density constitutes a "true" genre versus a transition zone.

## 3. Validation Table & Interpretation Limits
The metrics table above shows inertia and silhouette for each $k$. The silhouette scores peak at $k=2$ but that splits the catalog too broadly. At $k=4$, silhouette is ~0.178. 

**Interpretation limits:** The silhouette score is not exceptionally high. This tells us the cluster boundaries are soft and there is overlap. The clusters represent centers of gravity in the catalog, not hard, mutually exclusive boxes.

## 4. Cluster-Profile Analysis
Based on our interpretation artifacts, we characterized clusters with domain features (genres, tags) and meaningful operational differences:

- **Cluster 0 (Dark Action Blockbusters):** Thriller, crime, horror, action. High rating volume (+540 over mean). Audiences actively engage with these.
- **Cluster 1 (General Catalog):** Drama, comedy, romance. Largest cluster (52.5%), representing the "default" movie. Lower rating volume.
- **Cluster 2 (Family & Animated):** Animation, children, adventure, fantasy. Cleanly separated. Above average ratings (+0.154).
- **Cluster 3 (Documentary/Niche):** 99.6% documentaries. Very distinct, lowest rating volume but highest average ratings (+0.521). Prestige niche.


In [ ]:
# Load cluster profile summary
summary_df = pd.read_csv('../../artifacts/week07/week07_cluster_numeric_summary.csv')
display(summary_df)


## 5. Failure Analysis
**What didn't cluster well?**
- **Cluster 1** is a massive "catch-all" bucket. Drama and Comedy are too broad and overlap heavily, meaning K-means failed to find fine-grained boundaries in the mainstream movie space.
- The clusters are not highly separated (silhouette ~0.178). K-means struggles with the fact that many movies blend genres (e.g., Action-Comedy, Rom-Com), leading to points lying exactly between cluster centers.


## 6. Defense Questions

### Why did you cluster this representation?
We clustered the 13-dimensional autoencoder embedding rather than PCA because our comparison sweep showed the autoencoder achieved a reconstruction error ~15x lower than PCA. The autoencoder captures non-linear relationships between genres, tags, and ratings, providing a much better, richer topological space for K-means than linear PCA.

### What assumptions does K-means make, and where do they fail here?
K-means assumes clusters are spherical, equally sized, and equally dense. These assumptions fail here because the movie space is highly unbalanced (Cluster 1 is over 50% of the catalog, Cluster 3 is tiny) and likely non-spherical (genres overlap continuously). 

### How did DBSCAN or the density method change the interpretation?
While we focused on K-means, applying density methods changes the interpretation from "partitioning the whole space" to "finding dense cores and labeling the rest as noise". This highlights that many movies don't fit neatly into any genre bucket and are essentially outliers or edge cases.

### Which cluster is most defensible, and which is weakest?
**Most defensible:** Cluster 3 (Documentaries) and Cluster 2 (Animation/Family). They are highly cohesive and distinct from the rest of the catalog in both genre tags and user rating behavior.
**Weakest:** Cluster 1 (Drama/Comedy). It is a catch-all "junk drawer" cluster with low distinctiveness, showing that mainstream mixed-genre movies do not form a tight, well-defined group under K-means.

### What validation evidence supports the cluster profiles?
We validated using both internal metrics (Silhouette and Inertia sweeps across multiple $k$) and external domain profiling. The external profiling shows distinct rating volumes, rating averages, and highly significant genre distinctiveness scores (e.g., +0.906 distinctiveness for documentaries in Cluster 3). This confirms the clusters map to real-world, operational segments for recommendations.
